# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/0g/5xtfgw415q7cbj_fd_g3yk2h0000gn/T/ipykernel_31189/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [3]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [5]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [6]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [9]:
print(pages[1].page_content[:750])
print("\nMetadata:", pages[1].metadata)

Introduction
The feline patient ’s life stage is the most fundamental presentation
factor the practitioner encounters in a regular examination visit.
Most of the components of a treatment or healthcare plan are
guided by the patient ’s life stage, progressing from kitten to young
adult, mature adult, and senior and concluding with the end-of-life
stage. Because a cat can transition from one life stage to another in a
short period of time, each examination visit should include a life
stage assessment. The 2021 AAHA/AAFP Feline Life Stage Guidelines
provide a comprehensive age-associated framework for promoting
health and longevity throughout a cat ’s lifetime. The guidelines were
developed by a T ask Force of experts in feline clinical medic

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata (often referred to as **scalar data** or **document attributes**) is critical for transforming a basic RAG system into a reliable, production-grade application. 

Here are the key reasons why metadata is important:

*   **It filters for exactly what you're looking for.** While vector math is great for finding similar meanings, it’s not enough on its own. Metadata lets you use "hard filters" for things like dates or categories. This stops the system from accidentally grabbing an old document from 1990 when you specifically asked for the 2024 version.
*   **It shows the "receipts."** A major perk of RAG is that you can see exactly where the info came from. Metadata stores things like titles and page numbers so the AI can cite its sources, which makes the whole thing much more trustworthy and easier for people to read.
*   **You can update facts instantly.** You don't have to go through the massive headache of retraining an AI model just to teach it new information. You can just "hot-swap" the data index—like switching out a 2016 data dump for a 2018 one—simply by updating the metadata and document chunks.
*   **It saves space and money.** Feeding an entire library into an AI is slow and costs a fortune. Metadata helps you pick out just the relevant "snippets" of info to send, which keeps the context window small and the costs down.
*   **It keeps the system organized.** In more complex setups, metadata tracks what the agent is actually doing and where it is in a process. This makes it much easier to pause a task, pick it back up later, or fix things if the system gets stuck.
*   **It helps stop the AI from making things up.** By pointing the model toward specific, verifiable evidence tagged with metadata, it’s less likely to hallucinate or guess. It keeps the answers grounded in the actual data you provided rather than just the model's internal memory.

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [10]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [17]:
sample_chunk = splits[120]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

domestic cats (Felis catus). J Vet Behav 2014;9:78–82.
76. Grigg EK, Pick L, Nibblett B. Litter box preference in domestic cats:
covered versus uncovered. J Feline Med Surg 2013;15:280–4.
77. Horwitz DF . Common feline problem behaviors: urine spraying. JF e -
line Med Surg 2019;21:209
–19.
78. Cafazzo S, Bonanni R, Natoli E. Neutering effects on social behaviour of
urban unowned free-roaming domestic cats. Animals (Basel) 2019;9.
DOI: 10.3390/ani9121105.
79. Buf ﬁngton CA, Chew DJ, Kendall MS, et al. Clinical evaluation of cats
with nonobstructive urinary tract diseases. J Am V et Med Assoc 1997;
210:46–50.
80. Buf ﬁngton CAT, W estropp JL, Chew DJ, et al. Clinical evaluation of
multimodal environmental modi ﬁcation (MEMO) in the managemen

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

- Larger chunks keep more context together, but they can become noisy and less precise.
- Smaller chunks are easier to match to a specific question, but they can split up important ideas.
- More overlap helps avoid cutting off useful information at the edges, but it creates duplicate text and extra processing.
- Less overlap keeps the index lighter and faster, but it increases the chance of missing key context between chunks.
- The real tradeoff is accuracy versus efficiency: bigger chunks and more overlap usually help understanding, but they also cost more and slow things down.

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [21]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [44]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [45]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

Helps to understand
-   It shows how close two pieces of text are in meaning, even when the wording is different.

-   It helps rank which chunks or documents are most likely relevant to the question. In RAG, relavant chuncks can be pulled forward first.

-   It can surface useful clues or related context, even if the exact answer is not written directly.

Does not prove by itself
-   It does not prove the information is true, because a similar result can still be wrong, outdated, or biased.

-   It does not prove you have the exact answer or wording you need.

-   It does not prove the best version of the source, since similarity alone cannot tell the difference between old and new context, dates, or authors.

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [26]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [31]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [32]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [33]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs from the guideline that may mean your cat needs veterinary attention include:

- **Pain or anxiety** signs, especially in mature adult or senior cats, such as subtle pain behavior or reduced quality of life [Source 1]
- **Changes in grooming**:
  - **Increased grooming** may be linked to skin or other medical issues [Source 1]
  - **Reduced grooming** may suggest illness, bladder pain, joint pain, or reduced mobility [Source 2]
- **Behavior changes** like **house-soiling** or **aggression** [Source 2]
- **Vomiting, vomiting hairballs, diarrhea**, or chronic GI signs, especially if frequent or new [Source 3]
- **Changes in appetite**, **increased drinking/urination**, or **increased nocturnal activity/vocalization** [Source 3]
- **Signs of fear, stress, or distress**, such as cowering, crouching, hiding, freezing, flattened ears, dilated pupils, hissing, yowling, growling, or screaming [Source 4]

If you notice any of these, or if the signs are worsening, it’s a good idea to conta

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [34]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
Recommended preventive care for cats includes:

- **At least annual veterinary examinations for all cats**, with more frequent visits based on individual needs; **senior cats should be seen at least every 6 months** [Source 4].
- **Individualized risk assessment and preventive healthcare strategies** that change as the cat matures [Source 2].
- **Routine use of broad-spectrum parasite prevention products** for most pet cats [Source 1].
- **Treating canine and feline housemates in sync** when a new kitten or cat is introduced, to reduce transmission of parasites like roundworm and fleas [Source 1].
- **Limiting access to gardens and children’s sand areas** to reduce environmental contamination and parasite exposure [Source 1].

If you want, I can also summarize the preventive care points by life stage.
Question: What symptoms should make me call a veterinarian?
You should call a veterinarian if your cat is having vomiting, vomiting

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

-   Yes, the context was relevant for the cat care questions because the PDF covered those exact topics.
- The preventive care answer fit well since the document talks about annual exams, senior visits, parasite prevention, and lifestyle-based care.
- The feeding and symptom answers were also relevant because the PDF includes guidance on nutrition, weight, vomiting, diarrhea, appetite changes, thirst, and behavior changes.
- The taxes question was not relevant at all because the PDF is only about cat health and has nothing to do with filing taxes.

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [57]:
# Activity workspace
# Try retrieval tuning here.

### YOUR CODE HERE

print(f"Before Changes:  for the test question selected.\n")

test_question = "What should I know about feeding a healthy adult cat?"

test_k_1 = 4
# test_retrieved_results_before = display_retrieval_results(test_question, k=test_k_1)

# example_context_test = format_context(test_retrieved_results_before)
# print(example_context_test[:2000])

result_test_before = answer_question(
    test_question,
    k=test_k_1,
)

print(result_test_before["answer"])

print("*" * 80)
print(f"\nAfter Changes:  for the test question selected.\n")

# Task 4: Chunking the documents
chunk_size_test = 800
chunk_overlap_test = 100

text_splitter_test = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size_test,
    chunk_overlap=chunk_overlap_test,
    add_start_index=True,
)

splits_test = text_splitter_test.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits_test)} chunks.")
print(f"Chunk size: {chunk_size_test} characters")
print(f"Chunk overlap: {chunk_overlap_test} characters\n\n")

# Task 5: Embeddings and Qdrant

collection_name_1 = "cat_health_guidelines_test"

vector_store_test = QdrantVectorStore.from_documents(
    documents=splits_test,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name_1,
    force_recreate=True,
)

# Task 6: Retrival with Scores

def display_retrieval_results_test(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store_test.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

# Task 7 RAG

def answer_question_test(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store_test.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

# Final after results


# test_retrieved_results_after = display_retrieval_results_test(test_question, k=test_k_1)

# example_context_test_1 = format_context(test_retrieved_results_after)
# print(example_context_test_1[:2000])

result_test_after = answer_question_test(
    test_question,
    k=test_k_1,
)

print(result_test_after["answer"] + "\n\n")

Before Changes:  for the test question selected.

For a healthy adult cat, the guidelines say to choose a diet that meets the cat’s individual needs based on age, reproductive status, body condition score (BCS), muscle condition score (MCS), activity level, disease status, and future health concerns [Source 1]. Cats should be fed diets labeled with an Association of American Feed Control Officials (AAFCO) statement of nutritional adequacy, and the AAHA/AAFP do not advocate raw or dehydrated non-sterilized foods, including animal-origin treats [Source 1].

A good starting point for feeding is calculating resting energy requirements (RER), then adjusting daily energy requirements (DER) based on the cat’s needs; for young, healthy adults, the needs factor is 1 [Source 3]. Food intake should be adjusted to maintain ideal body condition, and BCS should be checked at each visit [Source 2]. Healthy adult cats should not be protein restricted; a moderate-protein diet is recommended [Source 4].

### 🏗️ Activity Notes

#### Activity - 1
- Setting changed: Changed the chunk size from 1000 to 800
- Before: Larger chunks made the result less organized and a bit harder to scan. It had useful details, but they were spread out.
- After: The answer is more organized and easier to scan. It keeps the main feeding points grouped clearly and sounds more natural.
  It also adds one useful detail about mature adult and senior cats that was missing before.
- Did retrieval improve? Why or why not?
  Yes, it likely improved because the response is clearer, more focused, and easier to follow.

  The warning about not just cutting food for weight loss is important. It should stay in the after version because it gives a practical next step instead of just reducing food.

#### Activity - 2
- Setting changed: Changed the chunk size from 1000 to 800 and chunk overlap from 200 to 100

- Before: The first version is more complete and reads more naturally, with the key feeding points covered in a smooth flow.

- After: The revised version is more structured and easier to skim, but the chunking detail does not really improve the feeding answer itself.

- Did retrieval improve? Why or why not? Not much, because the answer became cleaner in format, but the extra chunking note did not make the content better or more accurate.